# W05 tutorial — the CAPM Verdict Card

**ACFI20009 Financial Data — 26/27**  
Dr Zeming Li, University of Bristol

**This week you build the CAPM Verdict Card:** a one-page, three-exhibit judgment on the CAPM, built in five steps: load the panel, estimate betas, run the cross-section, sort into deciles, and state the verdict.

Work straight down the notebook. Each section explains what we are doing and why, then does it. By the last cell the artefact exists and runs.

> ### 🤖 Working with Gemini in this notebook
>
> Colab has Gemini built in. You are **encouraged** to use it — reading and
> directing AI-written code is one of the skills this unit is built around.
>
> **One hard rule:** never paste rows of CRSP data into Gemini (or any other
> AI tool). The licence forbids it. Describe the *schema* instead — column
> names, dtypes, what a row means — which is all an assistant needs to write
> correct code.

## Setup

Run these two cells once, top to bottom, before anything else.

In [ ]:
# Colab already has pandas, numpy, matplotlib, statsmodels, scipy and sklearn.
# This notebook additionally needs: linearmodels.
%pip install -q linearmodels

In [ ]:
# --- Data -------------------------------------------------------------------
# The unit's datasets are distributed through Blackboard. Download the
# `ACFI20009_data` folder once, put it in your own Google Drive, and this cell
# will find it. Nothing licensed is ever stored in the public repository, and
# the data never leaves your personal Drive.
from pathlib import Path

try:                                    # running on Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/ACFI20009/data")
except ModuleNotFoundError:             # running locally (convenor)
    DATA = Path("2627/data")

assert DATA.exists(), (
    f"Data folder not found at {DATA}.\n"
    "Download it from Blackboard and place it at MyDrive/ACFI20009/data"
)
print("Data:", DATA, "|", len(list(DATA.glob('*.parquet'))), "parquet files")

---

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.options.display.float_format = "{:.4f}".format

In [ ]:
crsp = pd.read_parquet(DATA / "crsp_monthly.parquet")   # US common stocks, NYSE/AMEX/NASDAQ
mkt  = pd.read_parquet(DATA / "market_monthly.parquet") # the VW index you built in W04
ff   = pd.read_parquet(DATA / "ff_factors_monthly.parquet")  # factors + rf, decimal monthly units

rf = ff[["date", "rf"]]

panel = crsp.merge(rf, on="date", how="inner", validate="many_to_one")
panel["exret"] = panel["ret"] - panel["rf"]

mkt = mkt.merge(rf, on="date", how="inner", validate="one_to_one")
mkt["exmkt"] = mkt["vwretd"] - mkt["rf"]

panel = panel.merge(mkt[["date", "exmkt"]], on="date", validate="many_to_one")
panel = panel.dropna(subset=["exret", "exmkt"])

print(f"{panel['permno'].nunique()} permnos, "
      f"{panel['date'].min().date()} to {panel['date'].max().date()}")
print(f"Mean market excess return: {mkt['exmkt'].mean():.4f} per month")

**Fix this in your head:** the factor file is already in *decimal monthly units* — no dividing by 100, no dividing by 12. Keep that mean market excess return (roughly 0.7–0.8% per month) visible: it is the number the CAPM promises per unit of beta, and section 8, *Verify the AI*, is built out of what happens when someone forgets the units.

### 1. One stock's beta, two ways

Beta is the slope of the regression of a stock's excess return on the market's. We estimate it for Apple (permno 14593) by regression, then confirm it is nothing more than a covariance ratio.

In [ ]:
import statsmodels.formula.api as smf

aapl = panel[panel["permno"] == 14593]

fit = smf.ols("exret ~ exmkt", data=aapl).fit()
beta_reg = fit.params["exmkt"]
beta_cov = aapl["exret"].cov(aapl["exmkt"]) / aapl["exmkt"].var()

print(f"Beta by regression:      {beta_reg:.3f}")
print(f"Beta as Cov/Var:         {beta_cov:.3f}")
print(f"Alpha: {fit.params['Intercept']:.4f}  (t = {fit.tvalues['Intercept']:.2f})")

The two numbers agree to machine precision — the OLS slope *is* $\widehat{\mathrm{Cov}}(R_i^e, R_M^e)/\widehat{\mathrm{Var}}(R_M^e)$. A picture makes the estimate honest:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.scatter(aapl["exmkt"], aapl["exret"], alpha=0.4, color="#0F6BBA")
ax.axline((0, fit.params["Intercept"]), slope=fit.params["exmkt"],
          color="#0F766E")                    # the regression line from `fit`
ax.set_xlabel("Market excess return")
ax.set_ylabel("Apple excess return")
ax.set_title("One beta = the slope of one cloud")
plt.show()

The cloud is wide — most of Apple's month-to-month risk is idiosyncratic. The slope is the only part the CAPM says gets paid.

### 2. Every stock's beta in five lines

Now scale up: one regression per permno, via split-apply-combine.

In [ ]:
counts = panel.groupby("permno")["exret"].size()
keep = counts[counts >= 36].index          # at least 36 months per stock
sub = panel[panel["permno"].isin(keep)]

def est_beta(g):
    return np.polyfit(g["exmkt"], g["exret"], 1)[0]  # slope of the OLS line

betas = sub.groupby("permno")[["exret", "exmkt"]].apply(est_beta)
mean_exret = sub.groupby("permno")["exret"].mean()

cs = pd.DataFrame({"beta": betas, "mean_exret": mean_exret}).dropna()
print(f"{len(cs)} stocks with estimated betas")
print(cs["beta"].describe())

This takes a few seconds — thousands of regressions are running. Before you read the summary above, work out what the average beta ought to be. (The value-weighted average is exactly 1 by construction; the equal-weighted mean lands slightly above 1 because small stocks have higher betas.) The tails beyond $\pm 3$ are estimation noise, not economics — they are the seed of the errors-in-variables story, and of the flat line in section 4.

The build starts here. `panel`, `mkt`, `sub` and `cs` are all needed below.

---

### 3. A defensive stock

Apple amplifies the market; plenty of stocks do the opposite. The same market-model regression, run for ExxonMobil (permno 11850) over the full sample, makes the contrast concrete. (If that permno is missing from your snapshot, any permno with a long history from `counts` behaves the same way.)

In [ ]:
xom = panel[panel["permno"] == 11850]

fit_x = smf.ols("exret ~ exmkt", data=xom).fit()
beta_x = fit_x.params["exmkt"]

print(f"beta = {beta_x:.2f}")
print(f"alpha = {fit_x.params['Intercept']:.4f}  (t = {fit_x.tvalues['Intercept']:.2f})")
print(f"Systematic share of variance (= R²): {fit_x.rsquared:.1%}")

- The beta comes out below 1 — Exxon is **defensive**: an energy major whose earnings track oil prices as much as the equity cycle, so it dampens market moves where Apple amplifies them.
- The systematic share of variance equals the regression $R^2$, because $\mathrm{Var}(R^e_i) = \beta_i^2\mathrm{Var}(R^e_M) + \mathrm{Var}(\varepsilon_i)$ and $R^2$ is the first term over the total. The rest is idiosyncratic — oil shocks, refinery outages, litigation — which diversification removes and the CAPM says goes unpaid.
- Check the alpha's **t-statistic**, not its sign: for most long histories it is statistically indistinguishable from zero. A positive point estimate alone proves nothing — the exact confusion planted in the lecture's AI checkpoint.

---

### 4. The cross-sectional test: is the SML too flat?

The cross-section `cs` — one beta and one average excess return per stock — is everything the classic test needs. Regressing `mean_exret` on `beta` with a constant gives the two numbers the CAPM makes a prediction about, each with a t-statistic.

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(cs["beta"])
sml = sm.OLS(cs["mean_exret"], X).fit()

print(f"Intercept: {sml.params['const']:>8.4f}   (t = {sml.tvalues['const']:.2f})")
print(f"Slope:     {sml.params['beta']:>8.4f}   (t = {sml.tvalues['beta']:.2f})")
print(f"CAPM says the slope should be about {mkt['exmkt'].mean():.4f}")

If the CAPM held, the intercept would be zero and the slope would equal the mean market excess return (~0.0075 per month). The slope comes out far below that — roughly a third of it (about 0.003) — alongside a positive intercept.

The same finding reads more clearly as a picture: sort the stocks into beta deciles with `pd.qcut`, plot each decile's average return against its average beta, and draw the CAPM-predicted line through the same axes.

In [ ]:
cs["beta_dec"] = pd.qcut(cs["beta"], 10, labels=False, duplicates="drop") + 1
dec = cs.groupby("beta_dec")[["beta", "mean_exret"]].mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.axline((0, 0), slope=mkt["exmkt"].mean(),
          color="#0F766E", linestyle="--")              # CAPM prediction
ax.axline((0, sml.params["const"]), slope=sml.params["beta"],
          color="#0F6BBA")                              # the data's fitted line
ax.scatter(dec["beta"], dec["mean_exret"], color="#0F6BBA", s=45, zorder=3)
ax.set_xlabel("Estimated beta (decile average)")
ax.set_ylabel("Mean monthly excess return")
ax.set_title("Dashed teal = CAPM prediction, solid blue = the data")
plt.show()

The verdict in one sentence: **the security market line is too flat** — high-beta stocks do not earn the premium the CAPM promises, and low-beta stocks earn more than it allows (Black, Jensen and Scholes 1972; the "betting against beta" trade of Frazzini and Pedersen 2014). Note `duplicates="drop"` in the `qcut` call and the `+ 1` (since `labels=False` returns bins 0–9).

---

### 5. Why flat is not (yet) a conviction

A flat estimated SML does **not** by itself prove the CAPM false. Before convicting the theory, audit the test: there is an econometric problem buried in it, it pushes the slope in a known direction, and this week's ladder carries two standard fixes for it.

The regressor in the cross-sectional test is the *estimated* beta, $\hat\beta_i = \beta_i + u_i$, not the true one. Classical measurement error in a regressor causes **errors-in-variables (attenuation) bias**: the estimated slope is biased towards zero by the reliability ratio $\sigma_\beta^2/(\sigma_\beta^2 + \sigma_u^2)$, and the intercept rises to compensate — so a flat line with a positive intercept is exactly what noisy betas would produce *even if the CAPM were true*. The two fixes are (1) **portfolio sorts** (Black, Jensen and Scholes 1972), which average away the noise in beta at the cost of compressing the cross-section, and (2) **Fama–MacBeth (1973)** monthly cross-sectional regressions, whose standard errors respect the cross-correlation of returns within a month. The damning fact from the lecture: after both fixes, the line stays flat — so the noise story, while real, is not the whole story. (Section 4 of the Week 05 Notes derives the bias formally; Notes Exercise 3 lets you simulate it.)

---

### 6. Replicate Fama–French (1992): the size deciles

Fama and French's headline is a pattern in average returns across size, and `crsp` is enough to reproduce it: each month, sort stocks into size deciles with `pd.qcut` on *lagged* market cap, take the equal-weighted average return per decile per month, then average over months and read the pattern in % per month.

In [ ]:
crsp = crsp.sort_values(["permno", "date"])
crsp["mktcap_lag"] = crsp.groupby("permno")["mktcap"].shift(1)  # lagged: no look-ahead

sizes = crsp.dropna(subset=["mktcap_lag", "ret"]).copy()
sizes["size_dec"] = (sizes.groupby("date")["mktcap_lag"]
                     .transform(lambda x: pd.qcut(x, 10, labels=False,
                                                  duplicates="drop"))
                     .astype(int) + 1)

size_pattern = (sizes.groupby(["date", "size_dec"])["ret"].mean()  # EW within decile
                .groupby("size_dec").mean() * 100)
print(size_pattern.round(2))

- Sorting on *same-month* market cap uses information unavailable at portfolio formation and mechanically links the sort variable to the realised return (a stock that rose this month has both a higher cap and a higher return): **look-ahead bias**, Week 04's villain. Hence `shift(1)` within permno — and note the `sort_values` first, or the shift scrambles.
- `duplicates="drop"` is required because many stocks can share a bin boundary; `labels=False` gives 0–9, hence `+ 1`.
- The smallest decile earns by far the most — around 2.8% per month equal-weighted, against roughly 1% for every other decile. The small-firm premium survives in our 1990–2022 snapshot, but with a modern twist: beyond decile 1 the gradient is nearly flat, so the effect is **concentrated in the very smallest stocks** rather than declining smoothly across all ten deciles as in Fama–French's 1963–1990 sample. Honest replication reports what changed as well as what survived; McLean and Pontiff (2016) document that anomalies generally weaken after publication.

---

### 7. Your first Fama–MacBeth regression

The second rung of the fix ladder. Attaching each stock's beta (from `cs`) to its stock-month rows in `sub`, setting a (permno, date) MultiIndex, and passing the result to `linearmodels.FamaMacBeth` with `cov_type="kernel"` runs one cross-sectional regression per month and averages the coefficients. Two coefficients come back, and the CAPM makes a prediction about each.

In [ ]:
from linearmodels import FamaMacBeth

fm_panel = sub.merge(cs[["beta"]], left_on="permno", right_index=True)
fm_panel = fm_panel.set_index(["permno", "date"])          # entity first, then time
fm_res = FamaMacBeth.from_formula("exret ~ 1 + beta", data=fm_panel).fit(
    cov_type="kernel")                                      # Newey–West on the lambdas
print(fm_res.params)
print(fm_res.tstats)

- The `beta` coefficient is the average monthly **price of beta risk**, $\bar\lambda_1$: it comes out well below the ~0.0075 market premium and statistically indistinguishable from zero — implication 2 fails. The intercept $\bar\lambda_0$ is positive and significant: zero-beta assets earn more than the risk-free rate, another violation.
- Returns are strongly correlated *across stocks* within a month (they all load on the market and on common factors). Pooled OLS treats every stock-month as an independent observation and therefore drastically overstates precision. Fama–MacBeth runs one cross-sectional regression per month and takes the standard error from the *time series* of monthly premia — 386 genuinely distinct monthly estimates rather than two million pretend-independent stock-month rows. `cov_type="kernel"` additionally applies Newey–West to that time series because the premia are persistent.

*Honest flag (and fast-finisher extension):* we used full-sample betas, which peek at the future. The canonical implementation re-estimates betas on rolling 60-month windows using only past data; the flat conclusion is robust to either choice. Implementing the rolling version is the recommended extension if you finish early.

---

### 8. Verify the AI

A student asked an AI assistant to "test whether high-beta stocks earn higher average excess returns using the unit's data". It produced the code below and this conclusion:

> "The intercept of the cross-sectional regression is large, positive and highly significant, and the mean excess return of the market is 0.96% per month. Beta is weakly priced. Overall, US stocks earned substantial excess returns regardless of beta over 1990–2022."

In [ ]:
# --- AI-generated code: do not trust yet ---
crsp = pd.read_parquet(DATA / "crsp_monthly.parquet")
ff = pd.read_parquet(DATA / "ff_factors_monthly.parquet")
mkt = pd.read_parquet(DATA / "market_monthly.parquet")
panel = crsp.merge(ff[["date", "rf"]], on="date", how="inner")
# convert the risk-free rate to a monthly decimal rate
panel["exret"] = panel["ret"] - panel["rf"] / 100 / 12

mkt = mkt.merge(ff[["date", "rf"]], on="date", how="inner")
mkt["exmkt"] = mkt["vwretd"] - mkt["rf"] / 100 / 12

panel = panel.merge(mkt[["date", "exmkt"]], on="date")

counts = panel.groupby("permno")["exret"].size()
sub = panel[panel["permno"].isin(counts[counts >= 36].index)]
betas = sub.groupby("permno")[["exret", "exmkt"]].apply(
    lambda g: np.polyfit(g["exmkt"], g["exret"], 1)[0])
cs = pd.DataFrame({"beta": betas,
                   "mean_exret": sub.groupby("permno")["exret"].mean()}).dropna()

import statsmodels.api as sm
print(sm.OLS(cs["mean_exret"], sm.add_constant(cs["beta"])).fit().summary())

The code runs without error and the logic of the test is sound. **It contains exactly one planted flaw** — a single wrong belief about the data, applied wherever the risk-free rate is used. It is worth hunting for before reading on: the next cell names it, traces what it does to every number downstream, and gives the two-line diagnostic that would have caught it.

The flaw is the unit conversion, applied in two places from the same wrong belief:

```python
panel["exret"] = panel["ret"] - panel["rf"] / 100 / 12
```

`ff_factors_monthly.parquet` is **already in decimal monthly units** (the data contract and the lecture both say so). The AI assumed an annualised percentage — the format of raw T-bill series on the web — and divided by 1,200. That shrinks a risk-free rate of, say, 0.002 per month to about 0.0000017: effectively zero. Consequences downstream:

- "Excess" returns are really **raw** returns, shifted up by the mean risk-free rate (~0.2% per month over this sample).
- The market "excess" return of 0.96% is really the *raw* market return (the true excess figure is roughly 0.75%), and the same upward shift feeds the cross-sectional intercept — hence "large, positive and highly significant".
- The *slope* on beta barely moves (a nearly constant shift affects the level, not the cross-sectional gradient), which is why the code still "finds" weak pricing of beta. Plausible headline, wrong numbers: the most dangerous kind of AI error.

The two-line diagnostic — look at the data before transforming it:

In [ ]:
print(ff["rf"].describe()[["mean", "min", "max"]])   # ~0.003/month (whole file): already decimal-monthly
print(f"After /100/12 the mean 'rf' would be {ff['rf'].mean() / 1200:.7f} — i.e. zero")

A monthly mean of ~0.003 (~3% a year, over the whole factor file) is exactly what a T-bill rate should look like in decimal monthly units. If the AI's belief were right, the raw values would be around 2–5, not 0.003. The comment in the AI's code ("convert ... to a monthly decimal rate") is the tell: it asserts a data format without checking it. **Rule: never let an AI (or yourself) apply a unit conversion without printing the series first.**

#### ⚠️ Common mistakes this week
1. **Risk-free-rate unit errors** — dividing the already-decimal-monthly `rf` by 100 and/or 12 (the planted flaw in section 8), or subtracting an annualised web-sourced rate from monthly returns. Print the series; check the magnitude.
2. **Mixed sample windows** — merging factors from one period with returns from another; the inner join silently drops rows and skews every mean. One window everywhere: 1990-01 to 2022-03.
3. **`pd.qcut` failures** — omitting `duplicates="drop"` (raises `ValueError` on tied boundaries) and forgetting that `labels=False` returns 0–9, not 1–10.
4. **`FamaMacBeth` index order** — the MultiIndex must be (entity, time) with a datetime time level; (time, entity) or string dates produce a cryptic error.
5. **Reading the time-series $R^2$ as "average return explained"** — $R^2$ measures the share of *variance* explained; average returns are alpha's job. A portfolio can have $R^2 = 0.95$ and an enormous alpha.